# Osoba 4 — Flask API
## Etap 5: /score, /stats, /health

**Kolejność uruchamiania:**
1. Uruchom komórkę 1 — instalacja Flask
2. Uruchom komórkę 2 — zapisuje plik `app.py`
3. Uruchom komórkę 3 — uruchamia serwer Flask w tle
4. Uruchom komórkę 4 — testy ręczne (scenariusze awarii)

Po uruchomieniu komórki 3, Spark (Osoba 3) może wystartować komórkę 4 tryb `api`.

## Komórka 1 — Instalacja zależności

In [1]:
import subprocess
result = subprocess.run(["pip", "install", "flask", "-q"], capture_output=True, text=True)
print(result.stdout or "Flask już zainstalowany")
print("✅ Gotowe — Flask dostępny")

Flask już zainstalowany
✅ Gotowe — Flask dostępny


## Komórka 2 — Zapisz plik app.py

Komórka używa `%%file` żeby zapisać kod Flask bezpośrednio do pliku `app.py`.
Następnie uruchomimy go jako osobny proces (Flask nie może działać w tej samej pętli co Jupyter).

In [1]:
%%file app.py
"""
Flask API — Osoba 4 — Nuclear Reactor Scoring
Endpointy: POST /score | GET /stats | GET /health
"""

from flask import Flask, request, jsonify
import statistics
from datetime import datetime

app = Flask(__name__)

# Historia wszystkich zdarzeń (w pamięci RAM — czyści się przy restarcie)
event_history = []

# =============================================================================
# REGUŁY SCORINGU — każda reguła to jeden warunek alarmowy
# Gdy warunek jest spełniony, do sumy ryzyka (score) dodawane są punkty.
# =============================================================================
RULES = [
    # --- CIŚNIENIE (norma: 14.5 – 16.0 MPa) ---
    {
        "id": "PRESSURE_CRITICAL_LOW",
        "description": "Ciśnienie poniżej 13 MPa — możliwy wyciek chłodziwa (LOCA)",
        "check": lambda d: d.get("pressure_mpa", 15) < 13.0,
        "points": 40,
    },
    {
        "id": "PRESSURE_WARN_LOW",
        "description": "Ciśnienie poniżej 14.5 MPa — zbliżanie się do strefy wyciekowej",
        "check": lambda d: 13.0 <= d.get("pressure_mpa", 15) < 14.5,
        "points": 15,
    },
    {
        "id": "PRESSURE_CRITICAL_HIGH",
        "description": "Ciśnienie powyżej 17 MPa — ryzyko rozsadzenia obiegu pierwotnego",
        "check": lambda d: d.get("pressure_mpa", 15) > 17.0,
        "points": 40,
    },
    {
        "id": "PRESSURE_WARN_HIGH",
        "description": "Ciśnienie powyżej 15.5 MPa — wzrost ponad normę",
        "check": lambda d: 15.5 < d.get("pressure_mpa", 15) <= 17.0,
        "points": 10,
    },
    # --- TEMPERATURA RDZENIA (norma: 290–315 °C) ---
    {
        "id": "TEMP_CRITICAL_HIGH",
        "description": "Temperatura rdzenia powyżej 340 °C — przegrzanie krytyczne, ryzyko stopienia",
        "check": lambda d: d.get("temp_hot_c", 305) > 340.0,
        "points": 45,
    },
    {
        "id": "TEMP_WARN_HIGH",
        "description": "Temperatura rdzenia powyżej 320 °C — zbliżanie się do granicy bezpieczeństwa",
        "check": lambda d: 320.0 < d.get("temp_hot_c", 305) <= 340.0,
        "points": 20,
    },
    # --- STRUMIEŃ NEUTRONOWY — moc reaktora (norma: 20–90%) ---
    {
        "id": "NEUTRON_FLUX_CRITICAL",
        "description": "Strumień neutronów powyżej 95% — reaktor na granicy maksymalnej mocy",
        "check": lambda d: d.get("neutron_flux_pct", 60) > 95.0,
        "points": 30,
    },
    {
        "id": "NEUTRON_FLUX_HIGH",
        "description": "Strumień neutronów powyżej 85% — wysoka moc, wzmożony nadzór",
        "check": lambda d: 85.0 < d.get("neutron_flux_pct", 60) <= 95.0,
        "points": 10,
    },
    # --- PRZEPŁYW CHŁODZIWA (norma: 80–100%) ---
    {
        "id": "FLOW_CRITICAL_LOW",
        "description": "Przepływ chłodziwa poniżej 40% — poważna awaria chłodzenia rdzenia",
        "check": lambda d: d.get("flow_pct", 100) < 40.0,
        "points": 45,
    },
    {
        "id": "FLOW_WARN_LOW",
        "description": "Przepływ chłodziwa poniżej 70% — degradacja chłodzenia",
        "check": lambda d: 40.0 <= d.get("flow_pct", 100) < 70.0,
        "points": 20,
    },
    # --- PROMIENIOWANIE (norma: 0.10–0.13 µSv/h) ---
    {
        "id": "RADIATION_CRITICAL",
        "description": "Promieniowanie powyżej 0.5 µSv/h — poważny wyciek radioaktywności",
        "check": lambda d: d.get("radiation_usvh", 0.12) > 0.50,
        "points": 35,
    },
    {
        "id": "RADIATION_ELEVATED",
        "description": "Promieniowanie powyżej 0.20 µSv/h — podwyższony poziom, obserwacja",
        "check": lambda d: 0.20 < d.get("radiation_usvh", 0.12) <= 0.50,
        "points": 15,
    },
    # --- MOC ELEKTRYCZNA ---
    {
        "id": "POWER_ZERO",
        "description": "Moc elektryczna = 0 MW — turbina odłączona lub całkowity blackout (SBO)",
        "check": lambda d: d.get("power_mwe", 600) < 10.0,
        "points": 30,
    },
]


def calculate_risk_score(data: dict) -> dict:
    """Sprawdza wszystkie reguły i zwraca ocenę ryzyka."""
    score = 0
    triggered_rules = []
    details = []

    for rule in RULES:
        try:
            if rule["check"](data):
                score += rule["points"]
                triggered_rules.append(rule["id"])
                details.append(rule["description"])
        except Exception:
            pass

    # Przeliczanie sumy punktów na poziom ryzyka
    if score <= 20:
        risk_level = "LOW"
    elif score <= 40:
        risk_level = "MEDIUM"
    elif score <= 70:
        risk_level = "HIGH"
    else:
        risk_level = "CRITICAL"

    return {
        "score": score,
        "risk_level": risk_level,
        "triggered_rules": triggered_rules,
        "details": details,
        "timestamp_api": datetime.utcnow().isoformat() + "Z",
    }


@app.route("/score", methods=["POST"])
def score():
    """
    Przyjmuje jedno zdarzenie z reaktora, zwraca ocenę ryzyka.
    Używany przez Spark (Osoba 3) — każde zdarzenie z Kafki tu trafia.

    Przykład żądania:
    {
        "reactor_id": "PWR-UNIT-01",
        "neutron_flux_pct": 60.0,
        "pressure_mpa": 15.1,
        "temp_hot_c": 305.0,
        "flow_pct": 100.0,
        "radiation_usvh": 0.12,
        "power_mwe": 600.0
    }
    """
    data = request.get_json(silent=True)
    if data is None:
        return jsonify({"error": "Brak danych JSON w żądaniu"}), 400

    result = calculate_risk_score(data)
    result["reactor_id"] = data.get("reactor_id", "UNKNOWN")

    # Zapis do historii (do /stats)
    event_history.append({
        "reactor_id":      result["reactor_id"],
        "score":           result["score"],
        "risk_level":      result["risk_level"],
        "triggered_rules": result["triggered_rules"],
        "pressure_mpa":    data.get("pressure_mpa"),
        "temp_hot_c":      data.get("temp_hot_c"),
        "radiation_usvh":  data.get("radiation_usvh"),
        "flow_pct":        data.get("flow_pct"),
        "received_at":     result["timestamp_api"],
    })

    # Log w terminalu
    level = result["risk_level"]
    if level in ("HIGH", "CRITICAL"):
        print(f"  [!!!] {level:8s} score={result['score']:3d} | "
              f"{result['reactor_id']} | Reguły: {result['triggered_rules']}")
    else:
        print(f"  [ OK] {level:8s} score={result['score']:3d} | "
              f"{result['reactor_id']} | "
              f"P={data.get('pressure_mpa','?'):.2f}MPa "
              f"T={data.get('temp_hot_c','?'):.1f}°C")

    return jsonify(result), 200


@app.route("/stats", methods=["GET"])
def stats():
    """Zwraca statystyki historyczne od startu API."""
    if not event_history:
        return jsonify({
            "total_events": 0,
            "message": "Brak zdarzeń od startu API — czekam na dane z reaktora"
        }), 200

    scores = [e["score"] for e in event_history]

    risk_distribution = {"LOW": 0, "MEDIUM": 0, "HIGH": 0, "CRITICAL": 0}
    for e in event_history:
        risk_distribution[e.get("risk_level", "LOW")] = \
            risk_distribution.get(e.get("risk_level", "LOW"), 0) + 1

    rule_counts = {}
    for e in event_history:
        for rule in e.get("triggered_rules", []):
            rule_counts[rule] = rule_counts.get(rule, 0) + 1

    top_rules = sorted(
        [{"rule": k, "count": v} for k, v in rule_counts.items()],
        key=lambda x: x["count"], reverse=True
    )[:10]

    return jsonify({
        "total_events":       len(event_history),
        "risk_distribution":  risk_distribution,
        "avg_score":          round(statistics.mean(scores), 2),
        "max_score":          max(scores),
        "min_score":          min(scores),
        "top_triggered_rules": top_rules,
        "recent_events":      event_history[-5:],
    }), 200


@app.route("/health", methods=["GET"])
def health():
    """Liveness check — używany przez Spark (Osoba 3)."""
    return jsonify({
        "status":            "ok",
        "service":           "Nuclear Reactor Scoring API — Osoba 4",
        "version":           "1.0",
        "events_processed":  len(event_history),
        "timestamp":         datetime.utcnow().isoformat() + "Z",
    }), 200


if __name__ == "__main__":
    print("=" * 60)
    print("  NUCLEAR REACTOR SCORING API — Osoba 4")
    print("=" * 60)
    print("  POST http://localhost:5000/score   — scoring zdarzenia")
    print("  GET  http://localhost:5000/stats   — statystyki")
    print("  GET  http://localhost:5000/health  — liveness check")
    print("=" * 60)
    app.run(host="0.0.0.0", port=5000, debug=False)

Overwriting app.py


## Komórka 3 — Uruchom serwer Flask w tle

Flask uruchamiamy jako **osobny proces** (subprocess), bo serwer HTTP blokuje
pętlę Jupytera — gdybyśmy wywołali `app.run()` bezpośrednio, notebook by zamarzł.

**Zatrzymanie:** uruchom komórkę 3b poniżej, lub zrestartuj kernel.

In [2]:
import subprocess, time, sys

flask_process = subprocess.Popen(
    [sys.executable, "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(f"Flask uruchomiony w tle | PID: {flask_process.pid}")
time.sleep(2)  # daj Flaskowi chwilę na start

if flask_process.poll() is None:
    print("✅ Flask działa — API dostępne na http://localhost:5000")
    print("   Spark (Osoba 3) może teraz uruchomić tryb 'api'")
else:
    out, _ = flask_process.communicate()
    print(f"❌ Flask nie wystartował (kod: {flask_process.returncode})")
    print(out)

Flask uruchomiony w tle | PID: 16445
✅ Flask działa — API dostępne na http://localhost:5000
   Spark (Osoba 3) może teraz uruchomić tryb 'api'


In [4]:
# Komórka 3b — ZATRZYMANIE FLASK (uruchom gdy chcesz wyłączyć API)
flask_process.terminate()
print("Flask zatrzymany.")

Flask zatrzymany.


## Komórka 4 — Testy ręczne (4 scenariusze awarii)

Sprawdzamy czy API poprawnie scoruje każdy typ zdarzenia z producenta.
Można uruchomić bez Kafki i Sparka — tylko Flask musi działać (komórka 3).

In [3]:
import requests, json

BASE = "http://localhost:5000"

def test(label, method, path, body=None):
    r = requests.request(method, BASE + path, json=body)
    data = r.json()
    print(f"\n{'='*55}")
    print(f"TEST: {label}")
    if "risk_level" in data:
        level = data['risk_level']
        icon = "🔴" if level == "CRITICAL" else "🟠" if level == "HIGH" else "🟡" if level == "MEDIUM" else "🟢"
        print(f"{icon} risk_level = {level} | score = {data['score']}")
        if data['triggered_rules']:
            print(f"   Reguły: {data['triggered_rules']}")
        else:
            print("   Brak wyzwolonych reguł (normalna praca)")
    else:
        print(json.dumps(data, indent=2, ensure_ascii=False))

# ── Health check ──────────────────────────────────────────────────────────
test("GET /health — czy API żyje?", "GET", "/health")

# ── 1. Normalna praca (LOW) ───────────────────────────────────────────────
test("POST /score — normalna praca (oczekiwane: LOW)", "POST", "/score", {
    "reactor_id": "PWR-UNIT-01",
    "neutron_flux_pct": 60.0,
    "pressure_mpa": 15.1,
    "temp_hot_c": 305.0,
    "flow_pct": 100.0,
    "radiation_usvh": 0.12,
    "power_mwe": 600.0,
})

# ── 2. Wyciek chłodziwa — LEAK (HIGH/CRITICAL) ───────────────────────────
test("POST /score — wyciek LEAK (oczekiwane: HIGH/CRITICAL)", "POST", "/score", {
    "reactor_id": "PWR-UNIT-01",
    "neutron_flux_pct": 60.0,
    "pressure_mpa": 11.5,    # < 13 MPa → PRESSURE_CRITICAL_LOW (+40)
    "temp_hot_c": 310.0,
    "flow_pct": 95.0,
    "radiation_usvh": 0.45,  # > 0.20 → RADIATION_ELEVATED (+15)
    "power_mwe": 600.0,
})

# ── 3. Awaria pompy — PUMP_FAIL (HIGH) ───────────────────────────────
test("POST /score — awaria pompy PUMP_FAIL (oczekiwane: HIGH)", "POST", "/score", {
    "reactor_id": "PWR-UNIT-01",
    "neutron_flux_pct": 60.0,
    "pressure_mpa": 15.1,
    "temp_hot_c": 335.0,     # > 320 → TEMP_WARN_HIGH (+20)
    "flow_pct": 25.0,        # < 40% → FLOW_CRITICAL_LOW (+45)
    "radiation_usvh": 0.13,
    "power_mwe": 600.0,
})

# ── 4. Całkowity blackout — TOTAL_BLACKOUT (CRITICAL) ────────────────────
test("POST /score — TOTAL_BLACKOUT (oczekiwane: CRITICAL)", "POST", "/score", {
    "reactor_id": "PWR-UNIT-01",
    "neutron_flux_pct": 60.0,
    "pressure_mpa": 15.1,
    "temp_hot_c": 360.0,     # > 340 → TEMP_CRITICAL_HIGH (+45)
    "flow_pct": 5.0,         # < 40 → FLOW_CRITICAL_LOW (+45)
    "radiation_usvh": 0.80,  # > 0.50 → RADIATION_CRITICAL (+35)
    "power_mwe": 0.0,        # < 10 → POWER_ZERO (+30)
})

# ── Statystyki końcowe ────────────────────────────────────────────────────
test("GET /stats — podsumowanie", "GET", "/stats")


TEST: GET /health — czy API żyje?
{
  "events_processed": 0,
  "service": "Nuclear Reactor Scoring API — Osoba 4",
  "status": "ok",
  "timestamp": "2026-05-30T16:53:44.148362Z",
  "version": "1.0"
}

TEST: POST /score — normalna praca (oczekiwane: LOW)
🟢 risk_level = LOW | score = 0
   Brak wyzwolonych reguł (normalna praca)

TEST: POST /score — wyciek LEAK (oczekiwane: HIGH/CRITICAL)
🟠 risk_level = HIGH | score = 55
   Reguły: ['PRESSURE_CRITICAL_LOW', 'RADIATION_ELEVATED']

TEST: POST /score — awaria pompy PUMP_FAIL (oczekiwane: HIGH)
🟠 risk_level = HIGH | score = 65
   Reguły: ['TEMP_WARN_HIGH', 'FLOW_CRITICAL_LOW']

TEST: POST /score — TOTAL_BLACKOUT (oczekiwane: CRITICAL)
🔴 risk_level = CRITICAL | score = 155
   Reguły: ['TEMP_CRITICAL_HIGH', 'FLOW_CRITICAL_LOW', 'RADIATION_CRITICAL', 'POWER_ZERO']

TEST: GET /stats — podsumowanie
{
  "avg_score": 68.75,
  "max_score": 155,
  "min_score": 0,
  "recent_events": [
    {
      "flow_pct": 100.0,
      "pressure_mpa": 15.1,
      "r

## Sprawdzamy czy dane napływają

In [4]:
import requests
r = requests.get("http://localhost:5000/stats", timeout=5)
print(r.json())

{'avg_score': 68.75, 'max_score': 155, 'min_score': 0, 'recent_events': [{'flow_pct': 100.0, 'pressure_mpa': 15.1, 'radiation_usvh': 0.12, 'reactor_id': 'PWR-UNIT-01', 'received_at': '2026-05-30T16:53:44.152517Z', 'risk_level': 'LOW', 'score': 0, 'temp_hot_c': 305.0, 'triggered_rules': []}, {'flow_pct': 95.0, 'pressure_mpa': 11.5, 'radiation_usvh': 0.45, 'reactor_id': 'PWR-UNIT-01', 'received_at': '2026-05-30T16:53:44.156453Z', 'risk_level': 'HIGH', 'score': 55, 'temp_hot_c': 310.0, 'triggered_rules': ['PRESSURE_CRITICAL_LOW', 'RADIATION_ELEVATED']}, {'flow_pct': 25.0, 'pressure_mpa': 15.1, 'radiation_usvh': 0.13, 'reactor_id': 'PWR-UNIT-01', 'received_at': '2026-05-30T16:53:44.159819Z', 'risk_level': 'HIGH', 'score': 65, 'temp_hot_c': 335.0, 'triggered_rules': ['TEMP_WARN_HIGH', 'FLOW_CRITICAL_LOW']}, {'flow_pct': 5.0, 'pressure_mpa': 15.1, 'radiation_usvh': 0.8, 'reactor_id': 'PWR-UNIT-01', 'received_at': '2026-05-30T16:53:44.163153Z', 'risk_level': 'CRITICAL', 'score': 155, 'temp_ho